# Date Format Validation
Checks date/timestamp columns for format consistency, auto-detects the best format, and flags suspicious date ranges.

**Configure the cell below**, then **Run All**. The final cell parses string dates — run it manually.

In [ ]:
# ── Configuration ──────────────────────────────────────────────
TABLE_NAME = "{{TABLE_NAME}}"
LAKEHOUSE_NAME = "{{LAKEHOUSE_NAME}}"
# Columns to validate as dates
DATE_COLUMNS = {{DATE_COLUMNS}}  # e.g., ["fecha_alta", "fecha_nacimiento"]
# Expected date format (Spanish default)
EXPECTED_DATE_FORMAT = "{{EXPECTED_DATE_FORMAT}}"  # e.g., "dd/MM/yyyy"
OUTPUT_SUFFIX = "_cleaned"
# ───────────────────────────────────────────────────────────────

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder.getOrCreate()
# Read from _cleaned if it exists (previous fixes), else original
try:
    df = spark.table(f"{TABLE_NAME}{OUTPUT_SUFFIX}")
    print(f"Reading from {TABLE_NAME}{OUTPUT_SUFFIX} (previous fixes applied)")
except:
    df = spark.table(TABLE_NAME)
    print(f"Reading from {TABLE_NAME} (original table)")
original_count = df.count()

# Common Spanish and international date formats to try
COMMON_FORMATS = [
    "yyyy-MM-dd", "dd/MM/yyyy", "dd-MM-yyyy",
    "yyyy-MM-dd HH:mm:ss", "dd/MM/yyyy HH:mm:ss",
    "d/M/yyyy", "yyyy/MM/dd", "yyyyMMdd",
]

print(f"Table: {TABLE_NAME}")
print(f"Rows: {original_count:,}")
print(f"Expected format: {EXPECTED_DATE_FORMAT}")

In [ ]:
print("=" * 60)
print("DATE FORMAT VALIDATION")
print("=" * 60)

best_formats = {}

for col_name in DATE_COLUMNS:
    field = [f for f in df.schema.fields if f.name == col_name][0]
    print(f"\n── Column: {col_name} (type: {field.dataType}) ──")

    if isinstance(field.dataType, (DateType, TimestampType)):
        null_dates = df.where(F.col(col_name).isNull()).count()
        total_non_null = original_count - null_dates
        print(f"  Already parsed as {field.dataType}. Non-null: {total_non_null:,}, Null: {null_dates:,}")

        if total_non_null > 0:
            date_stats = df.where(F.col(col_name).isNotNull()).agg(
                F.min(col_name).alias("min_date"),
                F.max(col_name).alias("max_date")
            ).collect()[0]
            print(f"  Range: {date_stats['min_date']} → {date_stats['max_date']}")

            suspect = df.where(
                (F.col(col_name) < F.lit("1900-01-01")) |
                (F.col(col_name) > F.current_date())
            ).count()
            if suspect > 0:
                print(f"  ⚠ {suspect:,} dates are before 1900 or in the future")
        continue

    if isinstance(field.dataType, StringType):
        non_null_df = df.where(
            F.col(col_name).isNotNull() & (F.trim(F.col(col_name)) != "")
        )
        non_null_count = non_null_df.count()

        if non_null_count == 0:
            print(f"  All values are null or empty — skipping.")
            continue

        formats_to_try = [EXPECTED_DATE_FORMAT] + [f for f in COMMON_FORMATS if f != EXPECTED_DATE_FORMAT]
        best_format = None
        best_success = 0

        for fmt in formats_to_try:
            parsed = non_null_df.withColumn("_parsed", F.to_date(F.col(col_name), fmt))
            success_count = parsed.where(F.col("_parsed").isNotNull()).count()
            if success_count > best_success:
                best_success = success_count
                best_format = fmt
            if success_count == non_null_count:
                break

        best_pct = round(best_success / non_null_count * 100, 2)
        print(f"  Best matching format: '{best_format}' ({best_pct}% parsed successfully)")
        best_formats[col_name] = best_format

        if best_pct < 100:
            failed_count = non_null_count - best_success
            print(f"  ⚠ {failed_count:,} values ({round(100 - best_pct, 2)}%) failed to parse")
            failed_df = non_null_df.withColumn(
                "_parsed", F.to_date(F.col(col_name), best_format)
            ).where(F.col("_parsed").isNull())
            print("  Sample unparseable values:")
            display(failed_df.select(col_name).distinct().limit(10))
        elif best_format != EXPECTED_DATE_FORMAT:
            print(f"  ℹ Format differs from expected '{EXPECTED_DATE_FORMAT}'")

---
## ⚠ Apply Fix — Parse String Dates
The cell below parses string date columns using the best-detected format.

**Review the analysis above before running this cell.**

In [ ]:
# ⚠ APPLY FIX — Run manually after review
cleaned_df = df

for col_name, fmt in best_formats.items():
    cleaned_df = cleaned_df.withColumn(col_name, F.to_date(F.col(col_name), fmt))

output_table = f"{TABLE_NAME}{OUTPUT_SUFFIX}"
cleaned_df.write.mode("overwrite").format("delta").saveAsTable(
    output_table
)

print(f"Parsed {len(best_formats)} date columns")
print(f"Output: {output_table}")